In [ ]:
import os
import warnings
import openpyxl
from openpyxl import Workbook, load_workbook
import pandas as pd
import xlrd

# Excel okurken çıkan gereksiz uyarıları gizle
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)


# ================== SABİTLER VE DOSYA YOLLARI ==================
girdi_klasoru = r""
hedef_dosya = r""
cikti_excel = r""

hedef_basliklar = [
    "Kuyu Adı", "Kuyu No", "Pafta No", "Zonu", "X (D)", "Y (K)",
    "Zemin Kotu", "Kuyu Derinliği (m)", "Tip (Arş./İşletme)", "İl",
    "İlçe", "Köy / Mahalle", "Açıldığı Yıl", "Havza Adı",
    "Temsil Ettiği Akifer", "OlcumMethodu", "OlcumAraligi",
    "Sezon", "Min. Ölçüm", "Max. Ölçüm tarihi", "Ölçüm adeti",
    "Dosya Adı", "Sayfa Adı"
]

# 2. Aşama için dinamik sütun indeksleri (0 tabanlı)
TOPLAM_SUTUN = len(hedef_basliklar)                     # 23 sütun
KUYU_NO_IDX = hedef_basliklar.index("Kuyu No")          # İndeks 1 (2. sütun)
SEZON_IDX = hedef_basliklar.index("Sezon")              # İndeks 17 (18. sütun)
OLCUM_ADETI_IDX = hedef_basliklar.index("Ölçüm adeti")  # İndeks 20 (21. sütun)

gormezden_gelinecek_sayfalar = [
    "Aylık Genel Bilgiler", "Aylık Grafik",
    "Genel Bilgi", "Genel Bilgi (2)", "Genel Bilgi (3)", "Genel Bilgi (4)",
    "Genel Bilgi (5)", "Genel Bilgiler (6)", "Lokasyon Haritası",
    "Mevsimlik Genel Bilgiler", "Mevsimlik Grafik",
]

ay_isimleri = {
    'ocak': 1, 'şubat': 2, 'subat': 2, 'mart': 3, 'nisan': 4,
    'mayıs': 5, 'mayis': 5, 'haziran': 6, 'temmuz': 7, 'ağustos': 8,
    'agustos': 8, 'eylül': 9, 'eylul': 9, 'ekim': 10, 'kasım': 11,
    'kasim': 11, 'aralık': 12, 'aralik': 12
}
# ===============================================================


# ================== YARDIMCI FONKSİYONLAR ==================
def olcum_periyodu_belirle(tarihler_serisi):
    """Bulunan ölçüm tarihlerine göre sıklığı Tek Ölçüm, Aylık veya Mevsimsel olarak hesaplar."""
    if len(tarihler_serisi) < 2:
        return "Tek Ölçüm"
    farklar = tarihler_serisi.sort_values().diff().dt.days.dropna()
    if len(farklar) == 0:
        return "Tek Ölçüm"

    medyan = farklar.median()
    if medyan <= 55:
        return "Aylık"
    else:
        return "Mevsimsel"


def is_valid_measurement(val):
    """Hücredeki değerin geçerli bir sayısal ölçüm olup olmadığını kontrol eder."""
    val_str = str(val).strip().lower()
    if val_str in ['nan', '', 'none', 'null', '-', 'veri yok', 'veriyok', 'kurudu', 'kuru']:
        return False
    try:
        float(val_str.replace(',', '.'))
        return True
    except ValueError:
        return False


def baslik_eslestir(hucre_metni):
    """Metni temizler ve esnek kelime araması ile hangi başlık olduğunu anlar."""
    if hucre_metni is None:
        return None

    # 1. Aşama: Türkçe karakterleri güvenli küçültme
    metin = str(hucre_metni).strip()
    turkce_harfler = {'İ': 'i', 'I': 'ı', 'Ö': 'ö', 'Ü': 'ü', 'Ş': 'ş', 'Ç': 'ç', 'Ğ': 'ğ'}
    for buyuk, kucuk in turkce_harfler.items():
        metin = metin.replace(buyuk, kucuk)

    metin = metin.lower().replace(':', '').replace('=', '')
    metin_bosluksuz = metin.replace(' ', '')

    if 'kuyuadı' in metin_bosluksuz or 'kuyuadi' in metin_bosluksuz:
        return "Kuyu Adı"

    # 2. Aşama: Kuyu No ve Pafta No ayırımı
    if 'kuyuno' in metin_bosluksuz or 'numarası' in metin_bosluksuz or 'sirano' in metin_bosluksuz:
        return "Kuyu No"
    if 'pafta' in metin_bosluksuz:
        return "Pafta No"

    if metin_bosluksuz == 'x' or 'x(d' in metin_bosluksuz or 'doğu' in metin_bosluksuz or 'sağa' in metin_bosluksuz:
        return "X (D)"
    if metin_bosluksuz == 'y' or 'y(k' in metin_bosluksuz or 'kuzey' in metin_bosluksuz or 'yukarı' in metin_bosluksuz:
        return "Y (K)"
    if 'zon' in metin_bosluksuz or 'dilim' in metin_bosluksuz:
        return "Zonu"

    if 'kot' in metin or 'zemin' in metin or metin_bosluksuz == 'z':
        return "Zemin Kotu"

    if 'derinli' in metin_bosluksuz:
        return "Kuyu Derinliği (m)"

    if 'tip' in metin or 'arş' in metin or 'işletme' in metin:
        return "Tip (Arş./İşletme)"

    if metin == 'il' or metin == 'ili':
        return "İl"
    if 'ilçe' in metin or 'ilce' in metin:
        return "İlçe"
    if 'köy' in metin or 'koy' in metin or 'mahalle' in metin:
        return "Köy / Mahalle"

    if 'açıldığı' in metin or 'acildigi' in metin or metin_bosluksuz == 'yıl':
        return "Açıldığı Yıl"
    if 'havza' in metin:
        return "Havza Adı"
    if 'akifer' in metin:
        return "Temsil Ettiği Akifer"
    if 'metod' in metin or 'limnigraf' in metin:
        return "OlcumMethodu"

    return None


def deger_al(satir):
    """Tekilleştirme aşamasında 'Ölçüm adeti' sütunundaki (indeks 20) sayısal değeri döndürür."""
    val = satir[OLCUM_ADETI_IDX]
    if val is None:
        return -999999
    try:
        return float(str(val).replace(',', '.'))
    except ValueError:
        return -999999
# ===========================================================


# ===============================================================
# 1. AŞAMA: HAM EXCEL DOSYALARINI TARAMA VE ÖZET TABLO OLUŞTURMA
# ===============================================================
wb_hedef = Workbook()
hedef_sayfa = wb_hedef.active
hedef_sayfa.title = 'Genel_Ozet'

sutun_indeksleri = {baslik: idx for idx, baslik in enumerate(hedef_basliklar, start=1)}
for baslik, idx in sutun_indeksleri.items():
    hedef_sayfa.cell(row=1, column=idx, value=baslik)

hedef_satir = 2

excel_dosyalari = [
    f for f in os.listdir(girdi_klasoru)
    if (f.endswith('.xlsx') or f.endswith('.xls')) and not f.startswith('~$')
]
print(f"Toplam {len(excel_dosyalari)} adet Excel dosyası işleniyor...\n")

for dosya_adi in excel_dosyalari:
    dosya_yolu = os.path.join(girdi_klasoru, dosya_adi)
    print(f"-> {dosya_adi} taranıyor...")

    is_xls = dosya_adi.endswith('.xls')

    if is_xls:
        wb = xlrd.open_workbook(dosya_yolu)
        sheet_names = wb.sheet_names()
    else:
        wb = openpyxl.load_workbook(dosya_yolu, data_only=True)
        sheet_names = wb.sheetnames

    for sayfa_adi in sheet_names:
        if sayfa_adi in gormezden_gelinecek_sayfalar:
            continue

        # Dosya tipine göre okuma köprüsü (Wrapper)
        if is_xls:
            kaynak_sayfa = wb.sheet_by_name(sayfa_adi)
            max_row = kaynak_sayfa.nrows
            max_column = kaynak_sayfa.ncols

            def get_val(r, c):
                if r > max_row or c > max_column or r < 1 or c < 1:
                    return None
                try:
                    val = kaynak_sayfa.cell_value(r - 1, c - 1)
                    return None if val == '' else val
                except IndexError:
                    return None
        else:
            kaynak_sayfa = wb[sayfa_adi]
            max_row = kaynak_sayfa.max_row
            max_column = kaynak_sayfa.max_column

            def get_val(r, c):
                return kaynak_sayfa.cell(row=r, column=c).value

        # Zaman serisi başlangıç satırını bulma
        yil_satiri, yil_sutunu, ay_sutunu = -1, -1, -1

        for r in range(1, 35):
            for c in range(1, 5):
                val = str(get_val(r, c)).strip().lower()
                if val in ['yil', 'yıl', 'yıllar']:
                    yil_satiri = r
                    yil_sutunu = c

        if yil_satiri != -1:
            for ac in range(yil_sutunu + 1, yil_sutunu + 4):
                aval = str(get_val(yil_satiri, ac)).strip().lower()
                if aval in ['ay', 'aylar']:
                    ay_sutunu = ac
                    break
            if ay_sutunu == -1:
                ay_sutunu = yil_sutunu + 1

        # Her bir kuyu (sütun) için işlem
        for col in range(5, max_column + 1):
            sutun_dolu_mu = any(get_val(r, col) is not None for r in range(1, 21))
            if not sutun_dolu_mu:
                continue

            # 1. Kısım: Meta verileri (üst bilgiler) çekme
            eslesme_sayisi = 0
            for row in range(1, 21):
                hedef_baslik = None

                for etiket_sutunu in [1, 2, 3, 4]:
                    hucre_degeri = get_val(row, etiket_sutunu)
                    hedef_baslik = baslik_eslestir(hucre_degeri)
                    if hedef_baslik:
                        break

                if hedef_baslik:
                    yazilacak_sutun = sutun_indeksleri[hedef_baslik]
                    veri = get_val(row, col)

                    if veri is None or str(veri).strip().lower() in ['', 'nan', 'none']:
                        for sag_col in range(col + 1, min(col + 4, max_column + 1)):
                            test_veri = get_val(row, sag_col)
                            if test_veri is not None and str(test_veri).strip().lower() not in ['', 'nan', 'none']:
                                veri = test_veri
                                break

                    hedef_sayfa.cell(row=hedef_satir, column=yazilacak_sutun, value=veri)
                    eslesme_sayisi += 1

            if eslesme_sayisi == 0:
                hedef_sayfa.cell(row=hedef_satir, column=1, value=get_val(1, col))
                hedef_sayfa.cell(row=hedef_satir, column=2, value=get_val(2, col))
                hedef_sayfa.cell(row=hedef_satir, column=3, value=get_val(3, col))
                hedef_sayfa.cell(row=hedef_satir, column=4, value=get_val(4, col))

            # 2. Kısım: Zaman serisi analizi
            gecerli_tarihler = []
            if yil_satiri != -1:
                mevcut_yil = None
                for r in range(yil_satiri + 1, max_row + 1):
                    y_val = get_val(r, yil_sutunu)
                    y_str = str(y_val).strip()

                    if y_val is not None and y_str.lower() not in ['nan', '', 'none']:
                        try:
                            y_int = int(float(y_str))
                            if 1900 < y_int < 2100:
                                mevcut_yil = y_int
                        except ValueError:
                            pass

                    if mevcut_yil is not None:
                        m_val = get_val(r, ay_sutunu)
                        m_str = str(m_val).strip().lower()
                        ay = None

                        if m_str in ay_isimleri:
                            ay = ay_isimleri[m_str]
                        else:
                            try:
                                ay = int(float(m_str))
                            except ValueError:
                                pass

                        if ay is not None and 1 <= ay <= 12:
                            olcum_degeri = get_val(r, col)
                            if is_valid_measurement(olcum_degeri):
                                gecerli_tarihler.append(pd.to_datetime(f"{mevcut_yil}-{ay:02d}-01"))

            if len(gecerli_tarihler) > 0:
                tarih_serisi = pd.Series(gecerli_tarihler)
                min_tarih = tarih_serisi.min().strftime('%m-%Y')
                max_tarih = tarih_serisi.max().strftime('%m-%Y')
                sezon = olcum_periyodu_belirle(tarih_serisi)
                adet = len(gecerli_tarihler)
            else:
                min_tarih, max_tarih, sezon, adet = "Veri Yok", "Veri Yok", "Veri Yok", 0

            hedef_sayfa.cell(row=hedef_satir, column=sutun_indeksleri["Min. Ölçüm"], value=min_tarih)
            hedef_sayfa.cell(row=hedef_satir, column=sutun_indeksleri["Max. Ölçüm tarihi"], value=max_tarih)
            hedef_sayfa.cell(row=hedef_satir, column=sutun_indeksleri["Sezon"], value=sezon)
            hedef_sayfa.cell(row=hedef_satir, column=sutun_indeksleri["Ölçüm adeti"], value=adet)
            hedef_sayfa.cell(row=hedef_satir, column=sutun_indeksleri["Dosya Adı"], value=dosya_adi)
            hedef_sayfa.cell(row=hedef_satir, column=sutun_indeksleri["Sayfa Adı"], value=sayfa_adi)

            hedef_satir += 1

cikti_klasoru = os.path.dirname(hedef_dosya)
if cikti_klasoru and not os.path.exists(cikti_klasoru):
    os.makedirs(cikti_klasoru)

wb_hedef.save(hedef_dosya)
print("\n========================= 1. AŞAMA TAMAMLANDI =========================")
print(f"Toplam {hedef_satir - 2} adet kuyu başarıyla işlendi.")
print(f"Özet Dosya: {hedef_dosya}\n")


# ===============================================================
# 2. AŞAMA: TEKRAR EDEN KAYITLARI FİLTRELEME VE TEMİZLEME
# ===============================================================
print("Veriler okunuyor ve 'Sezon' sütununa göre filtreleme kuralları uygulanıyor...")

wb_girdi = load_workbook(hedef_dosya)
ws_girdi = wb_girdi.active

# 23 sütunun tamamını oku
baslik_satiri = list(ws_girdi.iter_rows(min_row=1, max_row=1, max_col=TOPLAM_SUTUN, values_only=True))[0]

gruplar = {}
diger_satirlar = []

# Verileri Kuyu No (B sütunu / indeks 1) değerine göre grupla
for row in ws_girdi.iter_rows(min_row=2, max_col=TOPLAM_SUTUN, values_only=True):
    isim_b = row[KUYU_NO_IDX]

    if isim_b is None:
        diger_satirlar.append(row)
        continue

    if isim_b not in gruplar:
        gruplar[isim_b] = []

    gruplar[isim_b].append(row)

temiz_veriler = []

# Grupları Sezon sınıfına göre incele ve ölçüm adeti düşük olanları ele
for isim_b, satirlar in gruplar.items():
    if len(satirlar) == 1:
        temiz_veriler.extend(satirlar)
        continue

    aylik_grubu = []
    mevsimsel_grubu = []
    kalan_satirlar = []

    for satir in satirlar:
        sezon_degeri = str(satir[SEZON_IDX]).lower() if satir[SEZON_IDX] else ""

        if "aylık" in sezon_degeri or "aylik" in sezon_degeri:
            aylik_grubu.append(satir)
        elif "mevsimsel" in sezon_degeri or "mevsimlik" in sezon_degeri:
            mevsimsel_grubu.append(satir)
        else:
            kalan_satirlar.append(satir)

    if aylik_grubu:
        en_yuksek_aylik = max(aylik_grubu, key=deger_al)
        temiz_veriler.append(en_yuksek_aylik)

    if mevsimsel_grubu:
        en_yuksek_mevsimsel = max(mevsimsel_grubu, key=deger_al)
        temiz_veriler.append(en_yuksek_mevsimsel)

    temiz_veriler.extend(kalan_satirlar)

temiz_veriler.extend(diger_satirlar)

# Temizlenmiş verileri yeni Excel dosyasına yaz
print("Elenen satırlar çıkarıldı, yeni Excel dosyası oluşturuluyor...")

wb_cikti = Workbook()
ws_cikti = wb_cikti.active
ws_cikti.title = "Temizlenmis Veri"

ws_cikti.append(baslik_satiri)
for row in temiz_veriler:
    ws_cikti.append(row)

wb_cikti.save(cikti_excel)
print("\n========================= 2. AŞAMA TAMAMLANDI =========================")
print(f"Kalan toplam {len(temiz_veriler)} satır '{cikti_excel}' konumuna tüm 23 sütunuyla birlikte kaydedildi.")